---
title: "Statistics for Decisions"
subtitle: "From Excel to Python"
execute:
  echo: true
---

**In this lesson**

- Three numbers for a column: center, spread, and shape, in both tools
- Luck: what happens to the median when you only have 200 trips
- The standard error and a confidence interval, with the Excel twin
- Weekday against weekend: a t-test, and the same test on 30 trips a side
- What a p-value does and does not say, with a result that points two ways
- Two categories at once: the chi-square test on chapter 2's table
- The comparison the board cares about, and how to report "we cannot tell"
- One table that runs every test in the chapter: `generate_table1`
- Excel's `T.TEST`, `CHISQ.TEST`, `CONFIDENCE.T`, and where they stop

**The one idea**

> A difference between two numbers is a fact about this quarter. A statistical test asks a narrower question: if nothing were really different, how often would a quarter like this one show a gap this big by chance? The answer is permission to believe the difference. It is never the size of the difference, and the size is what the decision turns on.

::: {.content-visible when-format="html"}
[Download this notebook](chapter_07_lecture.ipynb){download="chapter_07_lecture.ipynb"} &nbsp;|&nbsp; [Open in Colab](https://colab.research.google.com/github/lshpaner/excel_to_python_exercises/blob/main/notebooks/chapter_07/chapter_07_lecture.ipynb){.colab-link}
:::

::: {.content-visible unless-format="html"}
Notebook and exercises for this chapter: `github.com/lshpaner/excel_to_python_exercises`, folder `notebooks/chapter_07/`.
:::

## Back to the decision

Chapter 6 ended with a question: the small multiples showed weekend trips running longer than weekday trips, and the chapter asked whether that gap is real or luck. Every number the board has been shown so far has been a comparison of some kind. Walk-up riders against pass holders. Weekend against weekday. This quarter against last year's, which is the comparison behind "ridership is falling". A comparison is two numbers and a gap, and the first question a careful reader asks about any gap is: compared to what, and how sure are you?

Statistics is the set of tools for the second half of that question. This chapter uses four of them, each on a comparison the memo already contains, and spends as much time on what the tools do not say as on what they do.

In [ ]:
try:
    import eda_toolkit
except ImportError:                      # Colab does not have it yet; local setups install it from requirements.txt
    %pip install -q eda_toolkit
    import eda_toolkit

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import StrMethodFormatter
from scipy import stats
from eda_toolkit import plot_distributions, normality_tests, generate_table1

LOCAL = "../../data/raw/"
URL = ("https://github.com/lshpaner/excel_to_python_exercises/"
       "releases/download/data/")
src = LOCAL if os.path.exists(LOCAL + "metro_trips.csv") else URL

trips = pd.read_csv(src + "metro_trips.csv")
trips["start_time"] = pd.to_datetime(trips["start_time"])
trips["date"] = trips["start_time"].dt.date
trips["day_type"] = np.where(trips["start_time"].dt.weekday >= 5, "weekend", "weekday")
trips = trips[(trips["duration"] > 1) & (trips["duration"] < 1440)].copy()   # chapter 3's keep mask

BLUE = "#1F4E79"
plt.rcParams["figure.figsize"] = (9, 4)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
len(trips)

`scipy.stats` is the new import. It is the statistics module of SciPy, which came along with pandas and matplotlib when you installed them, and it holds every test in this chapter.

## Three numbers for a column

Chapter 1 reported a mean and a median for duration and noticed they disagreed. Chapter 6 drew the histogram that explains why. Before testing anything, put the numbers on the table, because every test below is a question about one of them.

In [ ]:
d = trips["duration"]
pd.Series({"mean": d.mean(), "median": d.median(), "std": d.std(),
           "IQR": d.quantile(0.75) - d.quantile(0.25), "n": len(d)}).round(1)

Two measures of center and two of spread, and in each pair one is sensitive to the tail and one is not. The mean and the standard deviation (`std`) use every value, so the 1,300 trips over three hours drag both up: a standard deviation of 74 minutes for a column whose middle half runs from 7 to 27 is a number about the tail, not about a typical trip. The median and the interquartile range (`IQR`, the width of the middle half) ignore the tail entirely. In Excel these are `AVERAGE`, `MEDIAN`, `STDEV.S`, and `QUARTILE.INC(range, 3) - QUARTILE.INC(range, 1)`; `describe()` gives most of them in one call.

The shape matters for what comes later, because the oldest and most common tests assume something close to a bell curve. `normality_tests` runs the standard checks, and on this column the answer is not close.

In [ ]:
normality_tests(trips, features=["duration"])

Both tests say the column is not normal, which the histogram already said. The note about Shapiro-Wilk is worth reading: at 85,000 rows a normality test rejects anything, because no real column is exactly a bell curve and a big enough sample can see the gap. That is a preview of the chapter's main warning. A test's verdict depends on how much data you hand it, and more data makes every verdict harsher.

## Luck: what 200 trips would have told you

The whole quarter has 85,434 clean trips, so its median of 15 is not in doubt. But suppose Metro had surveyed 200 riders instead. Here are the medians and means of 1,000 random samples of 200 trips each, drawn from the real column with `sample`.

In [ ]:
#| label: fig-sampling
#| layout-ncol: 2
#| fig-cap: "Medians (left) and means (right) of 1,000 random samples of 200 trips each. The true median is 15; a sample of 200 gives anything from 12 to 18. The true mean is 29.3; the samples run from about 20 to over 50, and the spread is wider because the mean is at the mercy of whether the sample caught a few of the long trips."
draws = pd.DataFrame({
    "sample median": [d.sample(200, random_state=i).median() for i in range(1000)],
    "sample mean":   [d.sample(200, random_state=i).mean() for i in range(1000)],
})
common = dict(plot_type="hist", density_function=None, stat="count", hist_color=BLUE, hist_edgecolor="white",
              y_axis_label="Samples", plot_mean=False, plot_median=False, figsize=(5.5, 3.6),
              custom_xlabels={"sample median": "minutes", "sample mean": "minutes"}, label_fontsize=11, tick_fontsize=9)
plot_distributions(draws, ["sample median"], binwidth=0.5,
                   custom_titles={"sample median": "Median of 200 trips, 1,000 draws"}, **common)
plot_distributions(draws, ["sample mean"], bins=25,
                   custom_titles={"sample mean": "Mean of 200 trips, 1,000 draws"}, **common)
draws.describe().round(1).loc[["mean", "std", "min", "max"]]

This is the fact under everything else in the chapter. A number computed from a sample moves when the sample changes, and the amount it moves is something you can measure. The standard deviation of those 1,000 sample means is about 5 minutes. There is a formula for it that does not need 1,000 draws: the standard deviation of the column divided by the square root of the sample size, $73.6 / \sqrt{200} = 5.2$, and it is called the **standard error** of the mean.

The standard error is the unit in which statistical questions are asked. "Is 36.7 different from 26.1?" has no answer on its own. "Is a gap of 10.6 minutes large compared with how much these means wobble?" does.

### A confidence interval

Turn the standard error around and you get a range. The quarter's mean is 29.3 minutes; its standard error, with 85,434 trips, is $73.6 / \sqrt{85{,}434} = 0.25$. A 95% confidence interval is the mean plus or minus about two standard errors.

In [ ]:
se = d.std() / np.sqrt(len(d))
lo, hi = d.mean() - 1.96 * se, d.mean() + 1.96 * se
print(f"mean {d.mean():.1f}, standard error {se:.2f}, 95% interval {lo:.1f} to {hi:.1f}")

In Excel, `CONFIDENCE.T(0.05, STDEV.S(range), COUNT(range))` returns the half-width, 0.49 here, and you add and subtract it from `AVERAGE(range)` yourself. The reading is the same in both tools: a quarter like this one, with riders like these, would have a mean trip somewhere between 28.8 and 29.8 minutes. Nothing about this quarter's average is uncertain. The point of computing the interval is to see how narrow it is, because the comparisons below will not all be this lucky.

## Weekday against weekend: the t-test

Now chapter 6's question. Weekend trips look longer. Here are the numbers.

In [ ]:
by_day = trips.groupby("day_type")["duration"].agg(["size", "mean", "median", "std"]).round(1)
by_day

Weekend trips average 36.7 minutes against 26.1 on weekdays, and the medians say the same thing at 19 against 13. The gap is 10.6 minutes in the mean. The t-test asks the standard-error question: given how much each group's mean wobbles, could a gap this size appear between two groups drawn from the same riders?

In [ ]:
weekday = trips.loc[trips["day_type"] == "weekday", "duration"]
weekend = trips.loc[trips["day_type"] == "weekend", "duration"]
result = stats.ttest_ind(weekend, weekday, equal_var=False)   # Welch's t-test: the groups may have different spreads
print(f"t = {result.statistic:.1f}, p = {result.pvalue:.1e}")

Two numbers come back. The **t statistic** is the gap measured in standard errors: 17.7 of them. The **p-value** is the probability of a gap at least that many standard errors wide if the two groups really had the same mean, and at 17.7 standard errors it is $10^{-70}$, a number with 69 zeros after the decimal point. The gap is real. Excel's `T.TEST(weekend_range, weekday_range, 2, 3)` returns the same p-value; the `2` means two-tailed and the `3` means unequal variances, which is the Welch version and the one to use by default.

`equal_var=False` deserves a sentence. The classic t-test assumes both groups have the same spread; Welch's version does not, costs nothing, and is right more often. There is no reason to use the classic one.

### The same question with 30 trips a side

The weekend gap is real. It is also easy to miss. Here is the identical test on random samples of 30 weekday and 30 weekend trips, five times.

In [ ]:
rows = []
for seed in range(5):
    a = weekday.sample(30, random_state=seed)
    b = weekend.sample(30, random_state=seed)
    p = stats.ttest_ind(b, a, equal_var=False).pvalue
    rows.append({"draw": seed + 1, "weekday mean": a.mean(), "weekend mean": b.mean(), "p-value": p})
pd.DataFrame(rows).round(3)

Same riders, same real gap, and the test clears the conventional 0.05 line in none of the five draws. In draws 3 and 4 the weekday sample happened to catch a long trip and the gap vanished; in draw 1 the weekend mean was 19 minutes above the weekday mean and the p-value was still 0.32, because with 30 trips a side the standard error is about 15 minutes and 19 is barely more than one of them.

Nothing in the data changed between this cell and the previous one. Only the sample size did. This is why "not significant" has to be read as "could not tell with this much data" and never as "no difference", and why the first thing to ask about any test result is the n behind it.

## What a p-value does and does not say

Here is a result that points two ways at once, from a comparison nobody asked for: do electric bikes get ridden for longer than standard ones?

In [ ]:
electric = trips.loc[trips["bike_type"] == "electric", "duration"]
standard = trips.loc[trips["bike_type"] == "standard", "duration"]
print(f"means    electric {electric.mean():.1f}  standard {standard.mean():.1f}")
print(f"medians  electric {electric.median():.1f}  standard {standard.median():.1f}")
print(f"p-value  {stats.ttest_ind(electric, standard, equal_var=False).pvalue:.1e}")

The p-value is one in ten million. The difference is "highly significant". And the mean says electric trips are shorter while the median says they are longer, by four minutes each way. Both are true: the standard bikes carry more of the three-hour trips, which pulls their mean up, and the electric bikes carry more of the 15-to-30-minute trips, which pushes their median up. The test compared means, because that is what a t-test does, and it answered honestly. It did not say which number matters, and it did not say the four-minute gap is worth anything to Metro.

Five things a p-value is not:

1. **It is not the probability that the finding is wrong.** A p-value of 0.04 does not mean a 4% chance the groups are the same. It means a 4% chance of a gap this big if they were the same, which is a statement about the data, not about the truth.
2. **It is not the size of the effect.** With 85,000 rows, a gap of one minute in the mean is significant. Report the gap, in the units the reader cares about, next to every p-value, and let the reader judge whether one minute matters.
3. **It is not a verdict on the question you meant to ask.** The t-test compared means. If the median is the right summary for a skewed column, and chapter 1 argued it is, then a test on medians is the right test. `stats.mannwhitneyu(weekend, weekday)` compares the whole distributions instead of the means and is the usual choice for columns shaped like this one. On the weekend question it agrees; on the bike-type question it would say electric trips are longer, the opposite sign.
4. **0.05 is a habit, not a law.** It is the line most fields use for "worth reporting", and nothing happens at 0.049 that does not happen at 0.051. State the p-value and let the reader apply their own line.
5. **One test per question.** Run twenty tests on twenty columns and one of them will clear 0.05 by luck, on average, with nothing behind it. Decide the question first, then run its test, and say how many you ran.

## Two categories at once: the chi-square test

The t-test compares a number across two groups. Chapter 2's first real finding was a table of two categories, pass type by bike type, and the question for a table is whether the rows differ in their split. The chi-square test answers it by comparing the counts you have with the counts you would see if pass type and bike type had nothing to do with each other.

In [ ]:
table = pd.crosstab(trips["passholder_type"], trips["bike_type"])
chi2, p, dof, expected = stats.chi2_contingency(table)
print(f"chi-square = {chi2:,.0f}, degrees of freedom = {dof}, p = {p:.1e}")
pd.concat([table, pd.DataFrame(expected, index=table.index, columns=table.columns).round(0).astype(int)],
          axis=1, keys=["observed", "expected if independent"])

Monthly Pass holders took 29,648 electric trips; if bike choice had nothing to do with pass type they would have taken about 23,900. The chi-square statistic adds up how far every cell sits from its expected count, and 7,039 is enormous for a table this size, so the p-value is zero to any precision Python prints. Excel's `CHISQ.TEST(observed_range, expected_range)` gives the same p-value, but you have to build the expected table yourself, with `ROWTOTAL * COLTOTAL / GRANDTOTAL` in every cell; `chi2_contingency` hands it back.

And the same caveat as before, in reverse: the test says the split differs across pass types. The normalized bar chart in chapter 6 says by how much, 56% electric for Monthly Pass against 24% for Walk-up, and that chart is the finding. The chi-square test is the footnote that says the chart is not noise.

## The comparison the board cares about

Every test so far was run on trips, and with 85,000 of them every gap was significant. The board's question is about days: is ridership different on weekends, and is it falling? Days are the unit, and a quarter has 90 of them.

In [ ]:
per_day = trips.groupby("date").size().rename("trips")
per_day.index = pd.to_datetime(per_day.index)
day_type = np.where(per_day.index.weekday >= 5, "weekend", "weekday")
per_day.groupby(day_type).agg(["size", "mean", "std"]).round(0)

Weekend days average 1,010 trips against 925 on weekdays, a gap of 85 trips a day, or 9%. The test:

In [ ]:
wk = per_day[day_type == "weekend"]
wd = per_day[day_type == "weekday"]
r = stats.ttest_ind(wk, wd, equal_var=False)
se_gap = np.sqrt(wk.var() / len(wk) + wd.var() / len(wd))
print(f"gap {wk.mean() - wd.mean():.0f} trips/day, standard error {se_gap:.0f}, t = {r.statistic:.2f}, p = {r.pvalue:.3f}")
print(f"95% interval for the gap: {wk.mean() - wd.mean() - 1.96 * se_gap:.0f} to {wk.mean() - wd.mean() + 1.96 * se_gap:.0f}")

A p-value of 0.07, and a confidence interval for the gap that runs from a few trips below zero to about 175 above it. With 26 weekend days, each one as variable as the rain and the holidays make it, the data cannot pin the weekend effect down. The line for the memo is neither "weekends are busier" nor "weekends are no different": weekend days ran about 9% busier this quarter, the quarter is too short to say whether that is the normal pattern, and four quarters of daily counts would settle it.

This is the chapter's second lesson and the one the board needs. The comparison that matters most, days against days and quarters against quarters, is the one with the least data behind it. Trips are plentiful and days are scarce, and "ridership is falling" is a claim about days.

## One table, every test

Clinical papers open with a "Table 1": every variable summarized for each group, with a p-value per row. `generate_table1` builds it in one call, choosing Welch's test for numeric columns and the chi-square test for categorical ones, which is every test in this chapter.

In [ ]:
table1 = generate_table1(trips[["duration", "bike_type", "day_type"]],
                         continuous_cols=["duration"], categorical_cols=["bike_type"],
                         groupby_col="day_type", value_counts=True, decimal_places=1,
                         drop_columns=["Mode", "Missing (n)", "Missing (%)", "Min", "Max"])
table1

Read it against the sections above: the duration row is the weekday-against-weekend t-test, and the bike type rows are a chi-square test showing that weekend riders choose electric bikes less often (42% against 47%). A table like this is the right appendix for the memo, because it shows every comparison in one place with its n, its size, and its p-value side by side, which is the order a reader should weigh them in.

## Excel's statistics, and where they stop

| Question | Excel | Python |
|---|---|---|
| Center and spread | `AVERAGE`, `MEDIAN`, `STDEV.S`, `QUARTILE.INC` | `describe()` |
| Standard error of the mean | `STDEV.S(r) / SQRT(COUNT(r))` | `d.std() / np.sqrt(len(d))` |
| Confidence interval | `CONFIDENCE.T(0.05, STDEV.S(r), COUNT(r))` | `stats.t.interval` or the two-standard-error rule |
| Two groups, a number | `T.TEST(r1, r2, 2, 3)` | `stats.ttest_ind(a, b, equal_var=False)` |
| Two groups, skewed number | no built-in | `stats.mannwhitneyu(a, b)` |
| Two categories | `CHISQ.TEST(observed, expected)` with the expected table built by hand | `stats.chi2_contingency(table)` |
| Many variables, one table | by hand, one formula per cell | `generate_table1` |

Everything in the first four rows is one formula in Excel and one line in Python, and the p-values agree to every digit. The handoff happens in the last three rows. Excel has no rank-based test, makes you construct the expected table for a chi-square test by hand, and has no way to run a test per column and collect the results, so a Table 1 for ten variables is fifty formulas and an afternoon. The sampling experiment at the top of the chapter, 1,000 samples of 200 rows, is a `for` loop in Python and is not something a spreadsheet can do at all.

## Back to the decision

The chapter started with chapter 6's question and the answer is yes: weekend trips are longer, by six minutes in the median, and no sample of this quarter could make that gap go away. It ends on the board's question, where the answer is "not with 26 weekend days". Both answers go in the memo, and the second one is the more useful, because it tells Metro what to measure next.

The method is the same for every comparison the memo makes: state the gap in the reader's units, state the n, run the test that matches the summary you reported, and write the sentence that all three support. The p-value is the last of the three, and the one the reader will care about least once the other two are on the page.

## Looking ahead

- **Worked exercise:** `chapter_07_guided.ipynb`. The three numbers by pass type; the 30-trip experiment on Walk-up against One Day Pass, where the t-test and the rank test disagree; a chi-square test of pass type against weekday.
- **On your own:** `chapter_07_solo.ipynb`. Is ridership falling inside the quarter? Daily trips in January against March, with the gap, its interval, and the sentence for the memo. Then a bootstrap interval for the Walk-up median.
- **Before the next chapter:** chapter 8 fits a line through trips per day. Look at the weekday and weekend means above and guess what a line would need to know about a day to predict its count.

## Appendix: reproducing this chapter {.unnumbered}

`chapter_07_lecture.ipynb` is generated from this page; exercises come from `tools/build_chapter_07_exercises.py`. Every test uses `scipy.stats`, the sampling experiment uses pandas `sample` with fixed seeds so the figures reproduce, and `generate_table1` and `normality_tests` come from `eda_toolkit`. All numbers are computed on clean rows (the chapter 3 `keep` mask) from `data/raw/`.